<a href="https://colab.research.google.com/github/romashko1977/skills-github-pages/blob/main/bot_qwen_2_%D0%BB%D1%83%D1%87%D1%88%D0%B8%D0%B9!.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==========================================
# SHORT SCALP v2.1 - УСТАНОВКА И КОНФИГУРАЦИЯ
# ==========================================

!pip install ccxt==4.4.69 requests pandas numpy ipywidgets nest_asyncio -q

import ccxt
import requests
import pandas as pd
import numpy as np
import time
import logging
import json
from datetime import datetime
from dataclasses import dataclass, field
from typing import Optional, List, Dict
from IPython.display import display, HTML, clear_output, Audio, Javascript
import nest_asyncio
import threading
import urllib.parse
import wave
import struct
import os
import subprocess

nest_asyncio.apply()

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger("ShortScalp")

# ===== КОНФИГУРАЦИЯ SHORT SCALP v2.1 =====
API_KEY = "mx0vglB0rxnBfzuNkz"
API_SECRET = "4ec68c01ef1d472c98635a71c66cd612"
COINGLASS_KEY = "b44e8fdbcf8f49918d59cd6aeca50a7c"

DRY_RUN = False
LEVERAGE = 20
POSITION_PCT = 0.07
MIN_SCORE = 5
MIN_RR = 0.8
MAX_POSITIONS = 5
SCAN_INTERVAL = 60
MIN_PUMP = 3.0
MIN_OI_MCAP = 0.3
MIN_EMA_DEV = 5.0
SL_BUFFER = 3.0
MIN_LIQ_VOLUME = 5000

# ===== TELEGRAM =====
TG_BOT_TOKEN = "8754962684:AAHm7Yqtv6Tgc11ayYeIB2gLEby4SMiQQS8"
TG_CHAT_ID = "265978355"
TG_ENABLED = True

# ===== КЭШИ =====
_oi_cache = {}
_oi_5m_cache = {}
_cvd_5m_cache = {}
_tg_sent = set()
_spike_alerted = set()

print("="*70)
print("🚀 SHORT SCALP v2.1 - КОНФИГУРАЦИЯ ЗАГРУЖЕНА")
print("="*70)
print(f"📊 Режим: {'DRY RUN (тест)' if DRY_RUN else 'REAL (реальные сделки)'}")
print(f"📈 Leverage: {LEVERAGE}x")
print(f"💰 Position: {POSITION_PCT*100}% от депозита")
print(f"🎯 Min Score: {MIN_SCORE}/12")
print(f"📊 Min R:R: {MIN_RR}:1")
print(f"🔍 Min Pump: {MIN_PUMP}%")
print(f"📱 Telegram: {'ON' if TG_ENABLED else 'OFF'}")
print("="*70)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.8/223.8 kB 14.5 MB/s eta 0:00:00
🚀 SHORT SCALP v2.1 - КОНФИГУРАЦИЯ ЗАГРУЖЕНА
📊 Режим: REAL (реальные сделки)
📈 Leverage: 20x
💰 Position: 7.000000000000001% от депозита
🎯 Min Score: 5/12
📊 Min R:R: 0.8:1
🔍 Min Pump: 3.0%
📱 Telegram: ON


In [2]:
# ==========================================
# MEXC FUTURES CONNECTOR
# ==========================================

class Mexc:
    def __init__(self):
        self.ex = ccxt.mexc({
            "apiKey": API_KEY,
            "secret": API_SECRET,
            "options": {"defaultType": "swap"},
            "enableRateLimit": True
        })
        self.ex.load_markets()
        print("✅ MEXC Futures подключен")

    def symbols(self):
        return [s for s in self.ex.symbols if s.endswith(":USDT") and "/USDT" in s]

    def ticker(self, sym):
        try:
            t = self.ex.fetch_ticker(sym)
            return {
                "price": t.get("last", 0),
                "pct": t.get("percentage", 0) or 0,
                "vol": t.get("quoteVolume", 0) or 0,
                "high": t.get("high", 0) or 0,
                "low": t.get("low", 0) or 0
            }
        except:
            return {}

    def orderbook_sell(self, sym, depth=20):
        try:
            ob = self.ex.fetch_order_book(sym, limit=depth)
            bv = sum(b[1] * b[0] for b in ob["bids"])
            av = sum(a[1] * a[0] for a in ob["asks"])
            return av / (bv + av) * 100 if (bv + av) > 0 else 50
        except:
            return 50

    def funding(self, sym):
        try:
            return self.ex.fetch_funding_rate(sym).get("fundingRate", 0) or 0
        except:
            return 0

    def ohlcv(self, sym, tf="5m", limit=100):
        try:
            return self.ex.fetch_ohlcv(sym, tf, limit=limit)
        except:
            return []

    def positions(self):
        try:
            return [p for p in self.ex.fetch_positions() if abs(float(p.get("contracts", 0))) > 0]
        except:
            return []

    def balance(self):
        try:
            b = self.ex.fetch_balance({"type": "swap"})
            for key in ['USDT', 'usdt']:
                info = b.get(key, {})
                if isinstance(info, dict):
                    for fld in ['free', 'total']:
                        v = float(info.get(fld, 0) or 0)
                        if v > 0:
                            return v
            for fld in ['free', 'total']:
                if fld in b and isinstance(b[fld], dict):
                    v = float(b[fld].get('USDT', 0) or 0)
                    if v > 0:
                        return v
            if 'info' in b:
                raw = b['info']
                if isinstance(raw, dict):
                    for k in ['availableBalance', 'crossWalletBalance', 'totalWalletBalance', 'balance', 'equity']:
                        if k in raw:
                            v = float(raw[k] or 0)
                            if v > 0:
                                return v
                elif isinstance(raw, list):
                    for item in raw:
                        if isinstance(item, dict) and item.get('currency', '').upper() == 'USDT':
                            for k in ['equity', 'availableBalance', 'crossWalletBalance', 'balance']:
                                if k in item:
                                    v = float(item[k] or 0)
                                    if v > 0:
                                        return v
            try:
                resp = self.ex.contractPrivateGetAccountAssets()
                if isinstance(resp, dict) and 'data' in resp:
                    for item in resp['data']:
                        if item.get('currency', '').upper() == 'USDT':
                            v = float(item.get('equity', 0) or item.get('availableBalance', 0) or 0)
                            if v > 0:
                                return v
            except:
                pass
            return 0
        except:
            return 0

    def open_short(self, sym, usdt):
        try:
            self.ex.set_leverage(LEVERAGE, sym)
            price = self.ex.fetch_ticker(sym)["last"]
            amt = self.ex.amount_to_precision(sym, usdt / price)
            order = self.ex.create_market_sell_order(sym, float(amt))
            log.info(f"SHORT {sym} | {amt} @ ~{price} | {usdt} USDT")
            return order
        except Exception as e:
            log.error(f"Err SHORT {sym}: {e}")
            return None

    def set_tp_sl(self, sym, tp, sl):
        try:
            if tp > 0:
                self.ex.create_order(sym, "TAKE_PROFIT_MARKET", "buy", None, None,
                    {"stopPrice": tp, "closePosition": True, "workingType": "CONTRACT_PRICE"})
                log.info(f" TP {sym}: {tp}")
            if sl > 0:
                self.ex.create_order(sym, "STOP_MARKET", "buy", None, None,
                    {"stopPrice": sl, "closePosition": True, "workingType": "CONTRACT_PRICE"})
                log.info(f" SL {sym}: {sl}")
        except Exception as e:
            log.error(f"Err TP/SL {sym}: {e}")

    def cvd_from_trades(self, sym, limit=200):
        try:
            trades = self.ex.fetch_trades(sym, limit=limit)
            if not trades:
                return 0
            buy_vol = sum(t['amount'] * t['price'] for t in trades if t['side'] == 'buy')
            sell_vol = sum(t['amount'] * t['price'] for t in trades if t['side'] == 'sell')
            total = buy_vol + sell_vol
            if total == 0:
                return 0
            return (sell_vol - buy_vol) / total * 1000
        except:
            return 0

    def oi_change_mexc(self, sym):
        try:
            candles_1h = self.ex.fetch_ohlcv(sym, '1h', limit=3)
            if candles_1h and len(candles_1h) >= 2:
                prev_v = candles_1h[-2][5] if len(candles_1h[-2]) > 5 else candles_1h[-2][4]
                curr_v = candles_1h[-1][5] if len(candles_1h[-1]) > 5 else candles_1h[-1][4]
                if prev_v > 0:
                    return (curr_v - prev_v) / prev_v * 100
            return 0
        except:
            return 0

print("✅ MEXC Connector готов")

✅ MEXC Connector готов


In [3]:
# ==========================================
# COINGLASS CONNECTOR (ИСПРАВЛЕНО)
# ==========================================

class CoinGlass:
    BASE = "https://open-api-v3.coinglass.com/api"

    def __init__(self):
        self.headers = {"coinglassSecret": COINGLASS_KEY}
        print("✅ CoinGlass инициализирован")

    def _get(self, path, params={}):
        try:
            url = f"{self.BASE}{path}"
            r = requests.get(url, headers=self.headers, params=params, timeout=10)
            d = r.json()
            return d.get("data") if d.get("success") else None
        except Exception as e:
            log.debug(f"CoinGlass API error: {e}")
            return None

    def oi_mcap(self, coin):
        """OI / Market Cap ratio"""
        data = self._get("/futures/coins-markets")
        if data:
            for item in data:
                if item.get("symbol", "").upper() == coin.upper():
                    oi = float(item.get("openInterest", 0))
                    mc = float(item.get("marketCap", 1))
                    return oi/mc if mc > 0 else 0
        return 0

    def oi_change_pct(self, coin):
        """✅ OI change % - 4 метода"""
        # Method 1: CoinGlass
        data = self._get("/futures/openInterest/ohlc-history",
            {"symbol": coin, "interval": "1h", "limit": 2})
        if data and len(data) >= 2:
            prev = float(data[-2].get("openInterest", 1))
            curr = float(data[-1].get("openInterest", 1))
            if prev > 0:
                change = ((curr - prev) / prev * 100)
                log.debug(f"OI {coin}: {change:.2f}% (CoinGlass)")
                return round(change, 2)

        # Method 2: Binance
        try:
            url = f'https://fapi.binance.com/fapi/v1/openInterestStatistics?symbol={coin}USDT&period=1h&limit=2'
            r = requests.get(url, timeout=5)
            if r.status_code == 200:
                data = r.json()
                if len(data) >= 2:
                    prev = float(data[0]['sumOpenInterest'])
                    curr = float(data[1]['sumOpenInterest'])
                    if prev > 0:
                        change = ((curr - prev) / prev * 100)
                        log.debug(f"OI {coin}: {change:.2f}% (Binance)")
                        return round(change, 2)
        except:
            pass

        # Method 3: MEXC holdVol
        try:
            sym_mx = f'{coin}_USDT'
            url = f'https://contract.mexc.com/api/v1/contract/ticker?symbol={sym_mx}'
            r = requests.get(url, timeout=5)
            d = r.json()

            if d.get('success') and d.get('data'):
                hv = float(d['data'].get('holdVol', 0))
                lp = float(d['data'].get('lastPrice', 0))
                curr_oi = hv * lp if hv > 0 and lp > 0 else 0

                if curr_oi > 0:
                    if coin in _oi_cache:
                        prev_oi, prev_t = _oi_cache[coin]
                        _oi_cache[coin] = (curr_oi, time.time())
                        if prev_oi > 0 and (time.time() - prev_t) < 7200:
                            change = ((curr_oi - prev_oi) / prev_oi * 100)
                            log.debug(f"OI {coin}: {change:.2f}% (MEXC)")
                            return round(change, 2)
                    _oi_cache[coin] = (curr_oi, time.time())
        except:
            pass

        # Method 4: MEXC kline volume
        try:
            url = f'https://contract.mexc.com/api/v1/contract/kline/{coin}_USDT?interval=Min60&limit=3'
            r = requests.get(url, timeout=5)
            d = r.json()

            if d.get('success') and d.get('data') and len(d['data']) >= 2:
                prev_v = float(d['data'][-2][5] if len(d['data'][-2]) > 5 else d['data'][-2][4])
                curr_v = float(d['data'][-1][5] if len(d['data'][-1]) > 5 else d['data'][-1][4])
                if prev_v > 0:
                    change = ((curr_v - prev_v) / prev_v * 100)
                    log.debug(f"OI {coin}: {change:.2f}% (MEXC kline)")
                    return round(change, 2)
        except:
            pass

        return 0.0

    def cvd_negative(self, coin):
        """✅ CVD negative - 3 метода"""
        # Method 1: Binance long/short
        try:
            url = f'https://fapi.binance.com/fapi/v1/globalLongShortAccountRatio?symbol={coin}USDT&period=5m&limit=1'
            r = requests.get(url, timeout=5)
            if r.status_code == 200:
                data = r.json()
                if data and len(data) > 0:
                    ratio = float(data[-1].get('longShortRatio', 0.5))
                    cvd = -(ratio - 1) * 1000
                    log.debug(f"CVD {coin}: {cvd:.0f} (Binance L/S)")
                    return round(cvd, 0)
        except:
            pass

        # Method 2: CoinGlass
        data = self._get("/futures/globalLongShortAccountRatio",
            {"symbol": coin, "interval": "5m"})
        if data and len(data) > 0:
            ratio = float(data[-1].get("longRate", 50))
            cvd = -(ratio - 50) * 10
            log.debug(f"CVD {coin}: {cvd:.0f} (CoinGlass)")
            return round(cvd, 0)

        # Method 3: MEXC rise/fall
        try:
            url = f'https://contract.mexc.com/api/v1/contract/ticker?symbol={coin}_USDT'
            r = requests.get(url, timeout=5)
            d = r.json()
            if d.get('success') and d.get('data'):
                rate = float(d['data'].get('riseFallRate', 0))
                cvd = -rate * 100
                log.debug(f"CVD {coin}: {cvd:.0f} (MEXC)")
                return round(cvd, 0)
        except:
            pass

        return 0

    def liq_zones(self, coin, price):
        """✅ TP/SL на основе ликвидаций + ATR"""
        tp, sl = None, None

        # Method 1: CoinGlass liquidation
        try:
            data = self._get("/futures/liquidation/detail",
                {"symbol": coin, "interval": "12h"})

            if data and isinstance(data, list):
                longs_below = [(float(l.get("price", 0)), float(l.get("volUsd", 0)))
                    for l in data if float(l.get("price", 0)) < price and float(l.get("volUsd", 0)) > MIN_LIQ_VOLUME]
                shorts_above = [(float(l.get("price", 0)), float(l.get("volUsd", 0)))
                    for l in data if float(l.get("price", 0)) > price and float(l.get("volUsd", 0)) > MIN_LIQ_VOLUME]

                if longs_below:
                    longs_below.sort(key=lambda x: x[1], reverse=True)
                    tp = longs_below[0][0]

                if shorts_above:
                    shorts_above.sort(key=lambda x: x[0])
                    sl = shorts_above[0][0] * 1.02
        except Exception as e:
            log.debug(f"LiQ zones error {coin}: {e}")

        # Method 2: MEXC kline S/R
        if tp is None or sl is None:
            try:
                url = f'https://contract.mexc.com/api/v1/contract/kline/{coin}_USDT?interval=Min60&limit=24'
                r = requests.get(url, timeout=5)
                d = r.json()

                if d.get('success') and d.get('data'):
                    lows = [float(x) for x in d['data'].get('low', []) if x]
                    highs = [float(x) for x in d['data'].get('high', []) if x]

                    if lows and tp is None:
                        tp = min(lows[-12:]) if len(lows) >= 12 else min(lows)

                    if highs and sl is None:
                        sl = max(highs[-6:]) * 1.02 if len(highs) >= 6 else max(highs) * 1.02
            except:
                pass

        # Method 3: ATR fallback
        if tp is None or sl is None:
            try:
                url = f'https://contract.mexc.com/api/v1/contract/kline/{coin}_USDT?interval=Min5&limit=50'
                r = requests.get(url, timeout=5)
                d = r.json()

                if d.get('success') and d.get('data'):
                    hs = d['data'].get('high', [])
                    ls = d['data'].get('low', [])
                    cs = d['data'].get('close', [])

                    if len(hs) >= 14:
                        trs = []
                        for i in range(1, 15):
                            h = float(hs[-i]) if hs[-i] else 0
                            l = float(ls[-i]) if ls[-i] else 0
                            c = float(cs[-i-1]) if i < len(cs) and cs[-i-1] else 0
                            trs.append(max(h-l, abs(h-c), abs(l-c)))

                        atr = sum(trs) / len(trs)

                        if tp is None:
                            tp = price - (atr * 2.5)
                        if sl is None:
                            sl = price + (atr * 1.5)
            except:
                pass

        # Final safety
        if tp is None or tp <= 0:
            tp = price * 0.94

        if sl is None or sl <= price:
            sl = price * (1 + SL_BUFFER / 100)

        rr = (price - tp) / (sl - price) if sl > price else 0

        log.info(f" LIQ {coin}: TP={tp:.6f} SL={sl:.6f} R:R={rr:.2f}")

        return tp, sl

print("✅ CoinGlass готов (liq_zones включён)")

✅ CoinGlass готов (liq_zones включён)


In [4]:
# ==========================================
# HELPER FUNCTIONS
# ==========================================

def calc_ema(candles, period=21):
    if not candles or len(candles) < period:
        return 0
    closes = pd.Series([c[4] for c in candles])
    return closes.ewm(span=period, adjust=False).mean().iloc[-1]

def is_parabolic(candles, hours=4):
    if not candles or len(candles) < 12:
        return False, 0
    n = min(len(candles), hours * 12)
    start = candles[-n][4]
    end = candles[-1][4]
    chg = (end - start) / start * 100 if start > 0 else 0
    return chg > 15, chg

def get_oi_change_5m(coin, mx_obj=None):
    """✅ OI 5m change - ИСПРАВЛЕНО"""
    import time as _t
    try:
        sym_mx = f'{coin}_USDT'
        url = f'https://contract.mexc.com/api/v1/contract/ticker?symbol={sym_mx}'
        r = requests.get(url, timeout=5)
        d = r.json()

        curr_oi = None
        if d.get('success') and d.get('data'):
            hv = float(d['data'].get('holdVol', 0))
            lp = float(d['data'].get('lastPrice', 0))
            if hv > 0 and lp > 0:
                curr_oi = hv * lp

        if curr_oi is None or curr_oi <= 0:
            try:
                url2 = f'https://fapi.binance.com/fapi/v1/openInterest?symbol={coin}USDT'
                r2 = requests.get(url2, timeout=5)
                if r2.status_code == 200:
                    d2 = r2.json()
                    curr_oi = float(d2.get('openInterest', 0)) * float(d2.get('price', 0))
            except:
                pass

        if curr_oi is None or curr_oi <= 0:
            return 0.0

        now = _t.time()

        if coin not in _oi_5m_cache:
            _oi_5m_cache[coin] = []

        _oi_5m_cache[coin].append((now, curr_oi))
        _oi_5m_cache[coin] = [(t, v) for t, v in _oi_5m_cache[coin] if now - t < 600]

        if len(_oi_5m_cache[coin]) < 2:
            return 0.0

        old_entry = None
        for t, v in _oi_5m_cache[coin][:-1]:
            if now - t >= 120:
                old_entry = (t, v)
                break

        if old_entry is None and len(_oi_5m_cache[coin]) >= 2:
            old_entry = _oi_5m_cache[coin][0]

        if old_entry:
            old_t, old_oi = old_entry
            if old_oi > 0:
                change_pct = (curr_oi - old_oi) / old_oi * 100
                log.debug(f"OI5m {coin}: {change_pct:.2f}%")
                return round(change_pct, 2)

        return 0.0

    except Exception as e:
        log.debug(f"OI5m error {coin}: {e}")
        return 0.0

def get_cvd_change_5m(coin, mx_obj=None):
    """✅ CVD 5m change - ИСПРАВЛЕНО"""
    import time as _t
    try:
        curr_cvd = None

        try:
            url = f'https://fapi.binance.com/fapi/v1/aggTrades?symbol={coin}USDT&limit=500'
            r = requests.get(url, timeout=5)
            if r.status_code == 200:
                trades = r.json()
                buy_v = sum(float(t['q']) * float(t['p']) for t in trades if not t['m'])
                sell_v = sum(float(t['q']) * float(t['p']) for t in trades if t['m'])
                curr_cvd = sell_v - buy_v
                log.debug(f"CVD5m {coin}: {curr_cvd:.0f} (Binance)")
        except:
            pass

        if curr_cvd is None and mx_obj is not None:
            try:
                sym = f'{coin}/USDT:USDT'
                trades = mx_obj.ex.fetch_trades(sym, limit=500)
                if trades:
                    buy_v = sum(t['amount'] * t['price'] for t in trades if t['side'] == 'buy')
                    sell_v = sum(t['amount'] * t['price'] for t in trades if t['side'] == 'sell')
                    curr_cvd = sell_v - buy_v
                    log.debug(f"CVD5m {coin}: {curr_cvd:.0f} (MEXC)")
            except:
                pass

        if curr_cvd is None:
            return 0.0

        now = _t.time()

        if coin not in _cvd_5m_cache:
            _cvd_5m_cache[coin] = []

        _cvd_5m_cache[coin].append((now, curr_cvd))
        _cvd_5m_cache[coin] = [(t, v) for t, v in _cvd_5m_cache[coin] if now - t < 600]

        if len(_cvd_5m_cache[coin]) < 2:
            return 0.0

        old_entry = None
        for t, v in _cvd_5m_cache[coin][:-1]:
            if now - t >= 120:
                old_entry = (t, v)
                break

        if old_entry is None and len(_cvd_5m_cache[coin]) >= 2:
            old_entry = _cvd_5m_cache[coin][0]

        if old_entry:
            old_t, old_cvd = old_entry
            avg_vol = (abs(curr_cvd) + abs(old_cvd)) / 2
            if avg_vol > 0:
                change_pct = (curr_cvd - old_cvd) / avg_vol * 100
                log.debug(f"CVD5m {coin}: {change_pct:.1f}")
                return round(change_pct, 1)

        return 0.0

    except Exception as e:
        log.debug(f"CVD5m error {coin}: {e}")
        return 0.0

def check_spike_alert(rows):
    """Check for sharp OI/CVD spikes"""
    alerts = []
    for r in rows:
        coin = r.get('c', '')
        oi5 = r.get('oi5m') or 0
        cvd5 = r.get('cvd5m') or 0
        if abs(oi5) > 5 or abs(cvd5) > 80:
            key = f"{coin}_{int(time.time() // 300)}"
            if key not in _spike_alerted:
                _spike_alerted.add(key)
                alerts.append(coin)
    return alerts

print("✅ Helper Functions готовы")

✅ Helper Functions готовы


In [5]:
# ==========================================
# TELEGRAM BOT (Non-blocking)
# ==========================================

def send_tg(coin, price, tp, sl, score, rr, pump, drop, ob, cvd, oi, fd):
    if not TG_ENABLED:
        return
    key = f"{coin}_{int(time.time() // 1800)}"
    if key in _tg_sent:
        return
    _tg_sent.add(key)
    tp_pct = (price - tp) / price * 100 if price > 0 else 0
    sl_pct = (sl - price) / price * 100 if price > 0 else 0
    msg = (
        f"🔴 SHORT #{coin}\n"
        f"💰 Entry: {price:.6f}\n"
        f"🎯 TP: {tp:.6f} (-{tp_pct:.1f}%)\n"
        f"🛑 SL: {sl:.6f} (+{sl_pct:.1f}%)\n"
        f"⚖️ R:R: {rr:.1f} | Score: {score}/12\n"
        f"📈 Pump: +{pump:.1f}% | Drop: -{drop:.1f}%\n"
        f"📊 OB: {ob:.0f}% | CVD: {cvd:.0f} | OI: {oi:+.1f}%\n"
        f"💱 Fund: {fd*100:.3f}%\n"
        f"🕒 {datetime.now().strftime('%H:%M:%S MSK')}"
    )
    def _send():
        try:
            enc = urllib.parse.quote(msg)
            url = f"https://api.telegram.org/bot{TG_BOT_TOKEN}/sendMessage?chat_id={TG_CHAT_ID}&text={enc}"
            subprocess.run(['curl', '-s', '--connect-timeout', '3', '--max-time', '5', url],
                          capture_output=True, timeout=6)
            log.info(f"TG sent: {coin}")
        except Exception as e:
            log.error(f"TG err: {e}")
    threading.Thread(target=_send, daemon=True).start()

print("✅ Telegram Bot готов (non-blocking)")

✅ Telegram Bot готов (non-blocking)


In [6]:
# ==========================================
# SOUND GENERATOR
# ==========================================

class SoundGenerator:
    @staticmethod
    def play_short_alert():
        """Звук при SHORT сигнале"""
        try:
            sample_rate = 44100
            duration = 0.6
            num_samples = int(sample_rate * duration)

            frequencies = [523.25, 659.25, 783.99, 1046.5, 783.99]
            note_duration = duration / len(frequencies)

            samples = []
            for i, freq in enumerate(frequencies):
                note_samples = int(sample_rate * note_duration)
                for j in range(note_samples):
                    t = (i * note_duration) + (j / sample_rate)
                    sample = 0.3 * np.sin(2 * np.pi * freq * t)
                    sample *= np.exp(-5 * (t - i * note_duration) / note_duration)
                    samples.append(sample)

            filename = '/tmp/short_alert.wav'
            with wave.open(filename, 'w') as wav_file:
                wav_file.setnchannels(1)
                wav_file.setsampwidth(2)
                wav_file.setframerate(sample_rate)
                for sample in samples:
                    value = int(sample * 32767)
                    wav_file.writeframes(struct.pack('h', value))

            display(Audio(filename, autoplay=True))

        except Exception as e:
            print(f"⚠️ Ошибка звука: {e}")

    @staticmethod
    def play_spike_alert():
        """Звук при SPIKE alert"""
        try:
            sample_rate = 44100
            duration = 0.4
            num_samples = int(sample_rate * duration)

            samples = []
            for i in range(num_samples):
                t = i / sample_rate
                frequency = 880 * np.exp(-2 * t)
                sample = 0.4 * np.sin(2 * np.pi * frequency * t)
                sample *= np.exp(-8 * t)
                samples.append(sample)

            filename = '/tmp/spike_alert.wav'
            with wave.open(filename, 'w') as wav_file:
                wav_file.setnchannels(1)
                wav_file.setsampwidth(2)
                wav_file.setframerate(sample_rate)
                for sample in samples:
                    value = int(sample * 32767)
                    wav_file.writeframes(struct.pack('h', value))

            display(Audio(filename, autoplay=True))

        except Exception as e:
            print(f"⚠️ Ошибка звука: {e}")

print("✅ Sound Generator готов")

✅ Sound Generator готов


In [7]:
# ==========================================
# MAIN SCREENER (ОБЪЕДИНЕННЫЙ)
# ==========================================

def opt_screener(loop=False, interval=60):
    mexc = Mexc()
    cg = CoinGlass()
    sound = SoundGenerator()
    iteration = 0

    while True:
        try:
            iteration += 1
            now = datetime.now().strftime('%H:%M:%S MSK')
            print(f'Scanning... {now} | Iter #{iteration}')

            pos = mexc.positions()
            pos_syms = set(p.get('symbol', '') for p in pos)
            n_pos = len(pos)
            bal = mexc.balance()

            try:
                all_tickers = mexc.ex.fetch_tickers()
            except Exception as e:
                print(f'fetch_tickers error: {e}')
                all_tickers = {}

            pumps = []
            for sym, t in all_tickers.items():
                if not sym.endswith(':USDT') or '/USDT' not in sym:
                    continue
                pct = t.get('percentage', 0) or 0
                vol = t.get('quoteVolume', 0) or 0
                price = t.get('last', 0) or 0
                if pct < 5 or vol < 100000 or price <= 0:
                    continue
                high = t.get('high', 0) or price
                low = t.get('low', 0) or price
                pumps.append({'sym': sym, 'coin': sym.split('/')[0], 'price': price,
                              'pct': pct, 'vol': vol, 'high': high, 'low': low})
            pumps.sort(key=lambda x: -x['pct'])
            pumps = pumps[:20]
            print(f'Found {len(pumps)} pump candidates')

            rows = []
            for p in pumps:
                sym = p['sym']
                coin = p['coin']
                price = p['price']
                tk = p
                try:
                    drop = (p['high'] - price) / p['high'] * 100 if p['high'] > 0 else 0
                    candles = mexc.ohlcv(sym)
                    ema = calc_ema(candles)
                    ema_dev = (price - ema) / ema * 100 if ema > 0 else 0
                    ob = mexc.orderbook_sell(sym)
                    fd = mexc.funding(sym)

                    oi = cg.oi_change_pct(coin)
                    cvd = cg.cvd_negative(coin)
                    oi5m = get_oi_change_5m(coin, mexc)
                    cvd5m = get_cvd_change_5m(coin, mexc)

                    if oi == 0 and cvd == 0 and oi5m == 0 and cvd5m == 0:
                        time.sleep(0.5)
                        oi = cg.oi_change_pct(coin)
                        cvd = cg.cvd_negative(coin)
                        oi5m = get_oi_change_5m(coin, mexc)
                        cvd5m = get_cvd_change_5m(coin, mexc)

                    para, para_chg = is_parabolic(candles)
                    tp_liq, sl_liq = cg.liq_zones(coin, price)
                    if sl_liq <= price:
                        sl_liq = price * (1 + SL_BUFFER / 100)
                    rr = (price - tp_liq) / (sl_liq - price) if sl_liq > price else 0

                    s = 0
                    if p['pct'] >= 10: s += 1
                    if oi > 5: s += 1
                    if p['vol'] > 50_000_000: s += 1
                    if drop >= 3: s += 1
                    if cvd < 0: s += 1
                    oim = cg.oi_mcap(coin)
                    if oim > MIN_OI_MCAP: s += 1
                    if ema_dev >= MIN_EMA_DEV: s += 1
                    if tp_liq > 0 and tp_liq < price: s += 1
                    if rr >= MIN_RR: s += 1
                    if fd > 0: s += 1
                    if ob > 50: s += 1
                    if para: s += 1

                    trap = (p['pct'] >= 15 and cvd < 0 and oi > 5)
                    ip = sym in pos_syms
                    why = []

                    if s >= 6 and rr >= 0.8:
                        act = 'SHORT'

                        sound.play_short_alert()

                        if trap:
                            why.append('TRAP')
                        if cvd < 0:
                            why.append('CVD-')
                        if ob > 55:
                            why.append(f'OB{ob:.0f}')
                        if fd > 0:
                            why.append('F+')
                        if drop >= 3:
                            why.append(f'D{drop:.0f}')

                        send_tg(coin, price, tp_liq, sl_liq, s, rr, p['pct'], drop, ob, cvd, oi, fd)

                        if not DRY_RUN and not ip and n_pos < 5:
                            usdt_size = bal * POSITION_PCT
                            if usdt_size >= 5:
                                order = mexc.open_short(sym, usdt_size)
                                if order:
                                    mexc.set_tp_sl(sym, tp_liq, sl_liq)
                                    log.info(f'OPENED SHORT {coin} | TP: {tp_liq:.5f} SL: {sl_liq:.5f}')
                                    n_pos += 1
                                    pos_syms.add(sym)
                                    why.append('ENTERED')
                        else:
                            log.info(f'DRY_RUN SHORT {coin} s={s} rr={rr:.1f} tp={tp_liq:.5f} sl={sl_liq:.5f}')

                    elif s >= 4 and rr >= 0.5:
                        act = 'WAIT'
                        if s < 6:
                            why.append(f's{s}')
                        if rr < 0.8:
                            why.append(f'rr{rr:.1f}')
                        if ip:
                            why.append('pos')
                    else:
                        act = 'SKIP'
                        if s < 4:
                            why.append(f's{s}')

                    rows.append({'c': coin, 'p': tk['pct'], 'd': drop, 's': s, 'rr': rr, 'tp': tp_liq, 'sl': sl_liq,
                                 'v': tk['vol'], 'ob': ob, 'cvd': cvd, 'oi': oi, 'fd': fd, 'a': act, 'w': why,
                                 'ip': ip, 'tr': trap, 'oi5m': oi5m, 'cvd5m': cvd5m})
                    print(f'   {coin} +{p["pct"]:.1f}% s={s} act={act}')
                except Exception as e:
                    print(f'  ERR {coin}: {e}')
                    continue

            rows.sort(key=lambda x: ({'SHORT': 0, 'WAIT': 1, 'SKIP': 2}[x['a']], -x['s'], -x['p']))

            css = '@keyframes blink{0%,100%{opacity:1}50%{opacity:0.3}} .blink{animation:blink 1s infinite} .arrow{font-size:18px;}'
            h = f'<style>{css}</style>'
            h += f'<div style="background:#0a0a15;color:#ddd;padding:12px;border-radius:8px;">'
            h += f'<h2 style="color:#0ff;margin:0;">SHORT SCALP SCREENER v3.2</h2>'
            h += f'<p style="color:#888;margin:4px 0;">{now} | Iter #{iteration} | Pos: {n_pos}/5 | Bal: {bal:.1f} U</p>'
            h += '<table style="width:100%;border-collapse:collapse;color:#ccc;font-size:12px;">'
            h += '<tr style="color:#888;">'
            for col in ['#', '', 'Coin', 'Pump', 'Drop', 'Scr', 'R:R', 'OB', 'CVD', 'OI', 'OI5m', 'CVD5m', 'Fund', 'TP', 'SL', 'Why']:
                al = 'right' if col in ['Pump', 'Drop', 'R:R', 'OB', 'CVD', 'OI', 'OI5m', 'CVD5m', 'Fund', 'TP', 'SL'] else 'left' if col in ['Coin', 'Why'] else 'center'
                h += f'<th style="padding:3px 5px;text-align:{al};">{col}</th>'
            h += '</tr>'

            ns = nw = 0
            for i, r in enumerate(rows[:20]):
                a = r['a']
                if a == 'SHORT':
                    bg = '#0a1f0a'
                    ac = '#00ff00'
                    ab = 'bold'
                    ns += 1
                    arrow = '<span class="blink arrow" style="color:#0f0;">&#x25BC; SHORT</span>' if not r['tr'] else '<span class="blink arrow" style="color:#ff0;">&#x25BC; TRAP</span>'
                elif a == 'WAIT':
                    bg = '#1f1f0a'
                    ac = '#ffaa00'
                    ab = 'bold'
                    nw += 1
                    arrow = '<span style="color:#fa0;">&#x23F3; WAIT</span>'
                else:
                    bg = '#0a0a15'
                    ac = '#444'
                    ab = 'normal'
                    arrow = '<span style="color:#444;">&mdash;</span>'

                st = '#0e0e1f' if i % 2 == 0 else '#0a0a15'
                bg = bg if a in ('SHORT', 'WAIT') else st
                pm = '<span style="color:#f80;font-size:10px;">POS</span>' if r['ip'] else ''
                wy = ' '.join(r['w'][:3])
                cv = '#0f0' if r['cvd'] < 0 else '#f44'
                obc = '#0f0' if r['ob'] > 55 else '#666'
                fc = '#0f0' if r['fd'] > 0 else '#666'
                sc = '#0f0' if r['s'] >= 6 else '#fa0' if r['s'] >= 4 else '#444'
                rc = '#0f0' if r['rr'] >= 1.5 else '#fa0' if r['rr'] >= 0.8 else '#444'
                oc = '#0f0' if r['oi'] > 5 else '#666'

                _oi5 = r.get('oi5m')
                _cvd5 = r.get('cvd5m')
                oi5c = '#0f0' if (_oi5 or 0) > 3 else '#f44' if (_oi5 or 0) < -3 else '#666'
                cvd5c = '#0f0' if (_cvd5 or 0) > 30 else '#f44' if (_cvd5 or 0) < -30 else '#666'

                h += f'<tr style="background:{bg};border-bottom:1px solid #1a1a2a;">'
                h += f'<td style="padding:3px 5px;text-align:center;">{i+1}</td>'
                h += f'<td style="padding:3px 5px;">{arrow}{pm}</td>'
                h += f'<td style="padding:3px 5px;font-weight:bold;cursor:pointer" onclick="navigator.clipboard.writeText(this.textContent)" title="Click to copy">{r["c"]}</td>'
                h += f'<td style="padding:3px 5px;text-align:right;color:#f66;">+{r["p"]:.1f}%</td>'
                h += f'<td style="padding:3px 5px;text-align:right;color:#5c6;">-{r["d"]:.1f}%</td>'
                h += f'<td style="padding:3px 5px;text-align:center;color:{sc};">{r["s"]}/12</td>'
                h += f'<td style="padding:3px 5px;text-align:right;color:{rc};">{r["rr"]:.1f}</td>'
                h += f'<td style="padding:3px 5px;text-align:right;color:{obc};">{r["ob"]:.0f}%</td>'
                h += f'<td style="padding:3px 5px;text-align:right;color:{cv};">{r["cvd"]:.0f}</td>'
                h += f'<td style="padding:3px 5px;text-align:right;color:{oc};">{r["oi"]:+.1f}%</td>'
                h += f'<td style="padding:3px 5px;text-align:right;color:{oi5c};">{(_oi5 or 0):+.1f}%</td>'
                h += f'<td style="padding:3px 5px;text-align:right;color:{cvd5c};">{(_cvd5 or 0):+.0f}</td>'
                h += f'<td style="padding:3px 5px;text-align:right;color:{fc};">{r["fd"]*100:.3f}%</td>'
                h += f'<td style="padding:3px 5px;text-align:right;font-size:10px;">{r["tp"]:.5f}</td>'
                h += f'<td style="padding:3px 5px;text-align:right;font-size:10px;">{r["sl"]:.5f}</td>'
                h += f'<td style="padding:3px 5px;text-align:left;color:#aaa;font-size:10px;">{wy}</td>'
                h += '</tr>'

            h += '</table>'

            spike_coins = check_spike_alert(rows)
            h += f'<p style="margin:6px 0 0;color:#666;">'
            h += f'<span style="color:#0f0;font-weight:bold;">&#x25BC; SHORT: {ns}</span> | '
            h += f'<span style="color:#fa0;">&#x23F3; WAIT: {nw}</span> | '
            h += f'SKIP: {len(rows)-ns-nw} | Total: {len(rows)}</p>'

            if spike_coins:
                h += f'<div class="blink" style="background:#300;color:#f00;padding:6px;margin:4px 0;border:2px solid #f00;border-radius:5px;font-weight:bold;">'
                h += f'SPIKE: {", ".join(spike_coins[:3])} | OI/CVD резкое изменение!</div>'
                sound.play_spike_alert()

            h += '</div>'

            if ns > 0 or spike_coins:
                h += '<script>(function(){var a=new(window.AudioContext||window.webkitAudioContext);[523.25,659.25,783.99,1046.5,783.99].forEach(function(f,i){var o=a.createOscillator(),g=a.createGain();o.connect(g);g.connect(a.destination);o.type="sine";o.frequency.value=f;g.gain.setValueAtTime(0.12,a.currentTime+i*0.12);g.gain.exponentialRampToValueAtTime(0.001,a.currentTime+i*0.12+0.15);o.start(a.currentTime+i*0.12);o.stop(a.currentTime+i*0.12+0.15)})})()</script>'

            clear_output(wait=True)
            display(HTML(h))

            if not loop:
                break
            time.sleep(interval)

        except KeyboardInterrupt:
            print('Stopped.')
            break
        except Exception as e:
            print(f'Error: {e}')
            import traceback
            traceback.print_exc()
            if not loop:
                break
            time.sleep(30)

print("✅ Main Screener готов")

✅ Main Screener готов


In [ ]:
# ==========================================
# ЗАПУСК SCREENER
# ==========================================

print("="*70)
print("🚀 SHORT SCALP v2.1 - ЗАПУСК")
print("="*70)
print(f"📊 Режим: {'DRY RUN' if DRY_RUN else 'REAL'}")
print(f"📈 Leverage: {LEVERAGE}x")
print(f"💰 Position: {POSITION_PCT*100}%")
print(f"🎯 Min Score: {MIN_SCORE}/12")
print(f"📱 Telegram: {'ON' if TG_ENABLED else 'OFF'}")
print("="*70)

# Запуск скринера
opt_screener(loop=True, interval=SCAN_INTERVAL)

#,,Coin,Pump,Drop,Scr,R:R,OB,CVD,OI,OI5m,CVD5m,Fund,TP,SL,Why
1,▼ SHORT,PUNCH,+20.6%,-3.9%,9/12,4.0,89%,-22,+1.4%,+4.0%,+200,0.011%,0.01006,0.01412,CVD- OB89 F+
2,▼ SHORT,MAGMA,+13.2%,-4.1%,8/12,5.8,47%,-15,-4.8%,-0.9%,-157,0.005%,0.06950,0.11669,CVD- F+ D4
3,▼ SHORTPOS,XPL,+32.0%,-1.2%,7/12,8.5,25%,-32,-5.4%,+20.3%,-16,0.005%,0.10770,0.15320,CVD- F+
4,▼ SHORT,BR,+15.7%,-11.3%,7/12,3.4,53%,-15,-0.2%,+0.9%,-60,0.005%,0.11458,0.16228,CVD- F+ D11
5,▼ SHORTPOS,STBL,+13.4%,-3.8%,7/12,2.5,58%,-13,+0.3%,+0.8%,+100,0.005%,0.02810,0.03502,CVD- OB58 F+
6,▼ SHORT,H,+10.8%,-0.0%,6/12,5.3,61%,-11,+0.6%,+2.8%,+187,0.005%,0.07730,0.08964,CVD- OB61 F+
7,▼ SHORTPOS,AIOT,+7.3%,-6.1%,6/12,2.8,56%,-7,+0.0%,+1.2%,-200,0.126%,0.01383,0.01976,CVD- OB56 F+
8,▼ SHORT,GUA,+7.0%,-10.5%,6/12,2.6,77%,-7,+0.1%,+0.6%,-200,0.017%,0.42663,0.48215,CVD- OB77 F+
9,▼ SHORT,FIGHT,+5.7%,-6.1%,6/12,1.7,70%,-6,+0.3%,+0.3%,+3,0.005%,0.00246,0.00312,CVD- OB70 F+
10,⏳ WAIT,PROMPT,+10.0%,-11.7%,6/12,0.7,56%,-10,+2.4%,+1.5%,-21,0.005%,0.02841,0.03668,rr0.7


Scanning... 20:32:37 MSK | Iter #84
Found 20 pump candidates


   XPL +33.4% s=7 act=SHORT


   PUNCH +21.0% s=9 act=SHORT


   BR +15.7% s=7 act=SHORT


   TAG +13.5% s=6 act=SHORT


   MAGMA +13.1% s=9 act=SHORT


   STBL +13.0% s=6 act=SHORT


   AKE +12.6% s=6 act=SHORT


   H +10.7% s=6 act=SHORT
